# L5.4 — The Transformer Block, End to End

Hands-on notebook for the lesson [`5-4-transformer-arch.mdx`](../../llm-quest-theory/level-5/5-4-transformer-arch.mdx).

> **Learning objectives**
> - Write a full `MiniGPT` with token embedding + sinusoidal PE + Pre-LN transformer blocks + tied head.
> - Account for every parameter and verify the total matches the theory's `L * 12 * d^2 + |V| * d` rule.
> - Plug in **RMSNorm** (LLaMA-style) as a drop-in replacement for LayerNorm.
> - Check that the output shape is `(B, T, |V|)` and that causal masking still holds.

## Connection to the theory
Covers **§1–§9** of the source `.mdx`. Everything Level 5 has shown you so far — tokenizer (5-1), embeddings (5-2), pretraining (5-3), attention (4-2), multi-head (4-4), positional encoding (4-3) — comes together here.

In [1]:
# ---- Setup ----
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cpu"
plt.rcParams["figure.dpi"] = 100
%matplotlib inline
print("torch", torch.__version__)

torch 2.2.2


## 1. Individual building blocks
Each piece is independently small; we build them in the order the theory lists them.

In [2]:
def sinusoidal_pe(T, d):
    pos = torch.arange(T).unsqueeze(1).float()
    i   = torch.arange(d).unsqueeze(0).float()
    div = 10000 ** ((i - i % 2) / d)
    return torch.where(i % 2 == 0, torch.sin(pos / div), torch.cos(pos / div))

class CausalMHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.h   = num_heads
        self.d_k = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.h, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
        attn   = scores.softmax(dim=-1)
        out    = attn @ v
        out    = self.proj(out.transpose(1, 2).contiguous().view(B, T, C))
        return (out, attn) if return_attn else out

class FFN(nn.Module):
    def __init__(self, d_model, d_ff=None, activation="gelu"):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU() if activation == "gelu" else nn.ReLU()
    def forward(self, x): return self.w2(self.act(self.w1(x)))

class RMSNorm(nn.Module):
    """LLaMA-style normalisation: divide by RMS(x), no centering."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps   = eps
    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.gamma * x / rms

class PreLNBlock(nn.Module):
    def __init__(self, d_model, num_heads, norm_cls=nn.LayerNorm):
        super().__init__()
        self.ln1 = norm_cls(d_model); self.attn = CausalMHA(d_model, num_heads)
        self.ln2 = norm_cls(d_model); self.ffn  = FFN(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

print("blocks defined")

blocks defined


## 2. Full `MiniGPT` with a tied output head

In [3]:
class MiniGPT(nn.Module):
    def __init__(self, vocab, d_model=128, n_heads=4, n_layers=4,
                 block_size=64, norm_cls=nn.LayerNorm, tied_head=True):
        super().__init__()
        self.vocab = vocab
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.register_buffer("pe", sinusoidal_pe(block_size, d_model))
        self.blocks  = nn.ModuleList([PreLNBlock(d_model, n_heads, norm_cls) for _ in range(n_layers)])
        self.ln_f    = norm_cls(d_model)
        # Head is either tied (reuse E.T) or a separate Linear
        self.tied_head = tied_head
        if not tied_head:
            self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, x):
        h = self.tok_emb(x) + self.pe[:x.shape[1]]
        for b in self.blocks: h = b(h)
        h = self.ln_f(h)
        if self.tied_head:
            return h @ self.tok_emb.weight.T
        return self.head(h)

VOCAB = 1024
model = MiniGPT(VOCAB, d_model=128, n_heads=4, n_layers=4, block_size=64).to(DEVICE)
x = torch.randint(0, VOCAB, (2, 40))
logits = model(x)
print("input shape :", x.shape)
print("logits shape:", logits.shape, "  (expected (2, 40, 1024))")

input shape : torch.Size([2, 40])
logits shape: torch.Size([2, 40, 1024])   (expected (2, 40, 1024))


## 3. Parameter accounting
The theory's rule of thumb: each block has `~12 * d_model^2` parameters. Embedding adds `|V| * d_model`. Let's verify empirically.

In [4]:
def count(m): return sum(p.numel() for p in m.parameters())

# Break the count down by component
embed_params = sum(p.numel() for n, p in model.named_parameters() if "tok_emb" in n)
block_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("blocks"))
final_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("ln_f"))
total = count(model)

print("By component:")
print(f"  embedding   :  {embed_params:>10,}")
print(f"  {len(model.blocks)} x transformer:  {block_params:>10,}   ({block_params // len(model.blocks):,} per block)")
print(f"  final LN    :  {final_params:>10,}")
print(f"  total       :  {total:>10,}")

# Theoretical estimates
d = 128
L = 4
rule_block  = 12 * d * d
rule_embed  = VOCAB * d
rule_total  = L * rule_block + rule_embed
print(f"\nRule-of-thumb estimates:")
print(f"  12 * d^2 per block    :  {rule_block:,}")
print(f"  {L} * 12 * d^2 + V*d  :  {rule_total:,}")
print(f"\nActual / estimate ratio:  {total / rule_total:.2f}")

By component:
  embedding   :     131,072
  4 x transformer:     793,088   (198,272 per block)
  final LN    :         256
  total       :     924,416

Rule-of-thumb estimates:
  12 * d^2 per block    :  196,608
  4 * 12 * d^2 + V*d  :  917,504

Actual / estimate ratio:  1.01


The match is not exact (the rule ignores biases and LNs) but it is close — and, crucially, it scales the right way. The moment you double `d_model`, parameters roughly quadruple.

## 4. Tied vs untied head — how many parameters does tying save?

In [5]:
tied_model   = MiniGPT(VOCAB, d_model=128, n_heads=4, n_layers=4, block_size=64, tied_head=True)
untied_model = MiniGPT(VOCAB, d_model=128, n_heads=4, n_layers=4, block_size=64, tied_head=False)
print(f"tied   : {count(tied_model):,} params")
print(f"untied : {count(untied_model):,} params")
print(f"savings: {count(untied_model) - count(tied_model):,} params (= |V| * d = {VOCAB * 128:,})")

tied   : 924,416 params
untied : 1,055,488 params
savings: 131,072 params (= |V| * d = 131,072)


## 5. RMSNorm drop-in
LLaMA replaces LayerNorm with RMSNorm — same shape, fewer operations. Swap in and check the output still has the right shape and the parameter count drops slightly (no bias, no mean).

In [6]:
rms_model = MiniGPT(VOCAB, d_model=128, n_heads=4, n_layers=4, block_size=64, norm_cls=RMSNorm)
x = torch.randint(0, VOCAB, (2, 40))
print("RMSNorm model output shape:", rms_model(x).shape)
print(f"LayerNorm params: {count(tied_model):,}")
print(f"RMSNorm   params: {count(rms_model):,}")
print("(RMSNorm has no bias and no mean -> slightly fewer parameters.)")

RMSNorm model output shape: torch.Size([2, 40, 1024])
LayerNorm params: 924,416
RMSNorm   params: 923,264
(RMSNorm has no bias and no mean -> slightly fewer parameters.)


## 6. Check causal masking holds end-to-end
Perturbing a token at position `t` must not change any output at positions `< t`. That is the property your training loop relies on when it computes loss across all positions at once.

In [7]:
tied_model.eval()
with torch.no_grad():
    x = torch.randint(0, VOCAB, (1, 20))
    logits_a = tied_model(x)
    x2 = x.clone(); x2[0, 12] = (int(x2[0, 12]) + 1) % VOCAB
    logits_b = tied_model(x2)

diff_per_pos = (logits_a - logits_b).abs().sum(dim=-1).squeeze(0).numpy()
for t, d in enumerate(diff_per_pos):
    marker = "<-- must be zero" if t < 12 else "(can change)"
    print(f"position {t:>2}: delta logits = {d:.3e}   {marker}")

position  0: delta logits = 0.000e+00   <-- must be zero
position  1: delta logits = 0.000e+00   <-- must be zero
position  2: delta logits = 0.000e+00   <-- must be zero
position  3: delta logits = 0.000e+00   <-- must be zero
position  4: delta logits = 0.000e+00   <-- must be zero
position  5: delta logits = 0.000e+00   <-- must be zero
position  6: delta logits = 0.000e+00   <-- must be zero
position  7: delta logits = 0.000e+00   <-- must be zero
position  8: delta logits = 0.000e+00   <-- must be zero
position  9: delta logits = 0.000e+00   <-- must be zero
position 10: delta logits = 0.000e+00   <-- must be zero
position 11: delta logits = 0.000e+00   <-- must be zero
position 12: delta logits = 1.183e+04   (can change)
position 13: delta logits = 5.375e+02   (can change)
position 14: delta logits = 4.767e+02   (can change)
position 15: delta logits = 4.106e+02   (can change)
position 16: delta logits = 3.932e+02   (can change)
position 17: delta logits = 4.388e+02   (can change

## 7. Feed-forward expansion factor
The theory notes that `d_ff = 4 * d_model` is the standard ratio (for good reason — it's where most of the model's capacity lives).

In [8]:
for d_model in [128, 256, 512]:
    block = PreLNBlock(d_model, num_heads=4)
    attn_params = sum(p.numel() for n, p in block.named_parameters() if "attn" in n)
    ffn_params  = sum(p.numel() for n, p in block.named_parameters() if "ffn"  in n)
    print(f"d_model={d_model}:  attn={attn_params:>8,}   ffn={ffn_params:>8,}  ratio ffn/attn = {ffn_params/attn_params:.2f}")

d_model=128:  attn=  66,048   ffn= 131,712  ratio ffn/attn = 1.99
d_model=256:  attn= 263,168   ffn= 525,568  ratio ffn/attn = 2.00
d_model=512:  attn=1,050,624   ffn=2,099,712  ratio ffn/attn = 2.00


FFN dominates the parameter count — roughly twice the attention. That's why "the model is mostly an MLP" is a surprisingly accurate mental model of a transformer block.

## 8. Quick checks

In [9]:
assert logits.shape == (2, 40, VOCAB)
# Tied head should save exactly |V| * d parameters compared to untied (small head params only).
assert count(untied_model) - count(tied_model) == VOCAB * 128
# Causal property: perturbation at position 12 must not affect 0..11
assert diff_per_pos[:12].max() < 1e-7, "causality is broken — output depends on future tokens"
# RMSNorm variant still works end-to-end
assert rms_model(torch.randint(0, VOCAB, (1, 10))).shape == (1, 10, VOCAB)
# The block parameter count should be within 30% of 12 * d^2 (theory estimate)
actual_per_block = block_params // len(model.blocks)
assert abs(actual_per_block - rule_block) / rule_block < 0.3, f"per-block count {actual_per_block:,} far from 12*d^2"
print("OK — model runs, parameters add up, causality and tying both hold.")

OK — model runs, parameters add up, causality and tying both hold.


## Reflection questions

1. The original transformer used Post-LN (`LN(x + sublayer(x))`). Modern GPT-2+ uses Pre-LN (`x + sublayer(LN(x))`). What breaks in Post-LN at 20+ layers — and what does Pre-LN buy you?
2. You can set `bias=False` in all Linears to shrink the count. Does that hurt quality on a tiny model like this? On a 100 B-param model? (It's a real design choice in LLaMA.)
3. The tied head saves `|V| * d` parameters. When would untying be worth it? (Hint: very large `d` and specialised output heads.)
4. Inference FLOPs per token ≈ `2 * N_params`. If our model has ~800 k parameters, how many FLOPs is each generated token, and how does that compare to a 7 B model?

## References
- Source theory: [`5-4-transformer-arch.mdx`](../../llm-quest-theory/level-5/5-4-transformer-arch.mdx)
- Next: [`5-5-sampling`](5-5-sampling.ipynb)